In [1]:
from nltk.corpus import stopwords

#stopwords.words('english')

In [2]:
from sqlalchemy import create_engine
import pandas as pd
import re
import string
import nltk
pd.set_option('display.max_colwidth', 100)

In [3]:
QUERY=""" 
SELECT
trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
,category_name "label"
FROM oneflare_reports.business_master bm
WHERE bm.total_quotes_made >= 3 and category_name is not null
"""

In [4]:
def make_engine():
    user = 'xxx'
    pw = 'xxx'
    db_name = 'snowplow'
    host = 'redshift.oneflare.io'
    port = 5439
    engine_string = "postgresql://%s:%s@%s:%s/%s" % (
        user, pw, host, port, db_name)
    engine = create_engine(engine_string)
    return engine

redshift = make_engine()
data=pd.read_sql(QUERY,redshift)


In [4]:
data.head()

,feature,label
0,"Matthew Lynch Pest Control Matthew Lynch Pest Control \r\nProfessional, Punctual & Affordable \r...",Pest Control
1,AB Electrics AB Electrics provide the very best electrical services at affordable rates.\nThe co...,Security System
2,Arundel Electrical Queensland Arundel Electrical\r\nSpecialises in domestic services. So whether...,Security System
3,Anytime Electrics Anytime Electrics providing the very best electrical contracting services.\r\n...,Security System
4,"Fotocam Pty Ltd Professional photography specialising in Corporate, Food,\r\nIndustrial, Product...",Photographer


In [5]:
stopwords = nltk.corpus.stopwords.words('english')
wn = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [6]:
def clean_text_by_porter_stem(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens if (word not in stopwords) and (word.isdigit() ==False) ]
    
    return text

def clean_text_by_wordnet_lemmatize(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [wn.lemmatize(word)  for word in tokens if (word not in stopwords) and (word.isdigit() ==False)] 
    
    return text

In [13]:
#data.drop('feature_cleaned', axis=1, inplace=True)

In [28]:
data['feature_cleaned_ps']=data['feature'].apply(lambda x: clean_text_by_porter_stem( ('' if x is None else x).lower()))
data['feature_cleaned_wd']=data['feature'].apply(lambda x: clean_text_by_wordnet_lemmatize( ('' if x is None else x).lower()))

In [29]:
data.head()

,feature,label,feature_cleaned_ps,feature_cleaned_wd
0,"Anaxim Accounting Solutions The company is a qualified, professional accountant who deals with a...",Accountant,"[anaxim, account, solut, compani, qualifi, profession, account, deal, array, account, issu, prov...","[anaxim, accounting, solution, company, qualified, professional, accountant, deal, array, accoun..."
1,RCS Websites Let RCS Websites breath life in to your existing website!\r\nOur websites are simpl...,SEO and SEM,"[rc, websit, let, rc, websit, breath, life, exist, websit, websit, simpl, clean, profession, sur...","[rcs, website, let, rcs, website, breath, life, existing, website, website, simple, clean, profe..."
2,Intouch Carpet Cleaning Services InTouch Carpet and Pest Control is a family owned business who ...,Pest Control,"[intouch, carpet, clean, servic, intouch, carpet, pest, control, famili, own, busi, valu, old, f...","[intouch, carpet, cleaning, service, intouch, carpet, pest, control, family, owned, business, va..."
3,Shannon Thomas Carpenter Licensed Carpenter\r\n - Small Decks and Pergolas as well as re deck...,Carpenter,"[shannon, thoma, carpent, licens, carpent, small, deck, pergola, well, deck, repair, attent, det...","[shannon, thomas, carpenter, licensed, carpenter, small, deck, pergola, well, decking, repair, a..."
4,"M&M Cabinets Custom-made Cabinets for kitchen, laundries, bathroom and anywhere in your house. 3...",Builder,"[mm, cabinet, custommad, cabinet, kitchen, laundri, bathroom, anywher, hous, 3d, comput, draw, a...","[mm, cabinet, custommade, cabinet, kitchen, laundry, bathroom, anywhere, house, 3d, computer, dr..."


In [31]:
print(type(data.iloc[0]['feature_cleaned_ps']))
print(type(data.iloc[0]['feature_cleaned_wd']))

<class 'list'>
<class 'list'>


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [8]:
tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
tfidf_vectorized_matrix_ps = tfidf_vectorizer_ps.fit_transform(data['feature'] )



In [11]:
count_vectorizer_ps = CountVectorizer(analyzer=clean_text_by_porter_stem)
count_vectorized_matrix_ps = count_vectorizer_ps.fit_transform(data['feature'] )

In [9]:
tfidf_vectorized_matrix_ps.shape

(39002, 48679)

In [21]:
tfidf_vectorized_df_ps = pd.DataFrame(tfidf_vectorized_matrix_ps.toarray())

MemoryError: 

In [18]:
tfidf_vectorized_df_ps.head()

,0,1,2,3,4,5,6,7,8,9,...,48564,48565,48566,48567,48568,48569,48570,48571,48572,48573
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])

In [10]:
tfidf_vectorized_matrix_wn.shape

(38974, 56825)

In [ ]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])
tfidf_vectorized_df_wn = pd.DataFrame(tfidf_vectorized_matrix_wn.toarray())

In [19]:
tfidf_vectorized_df_ps_h = tfidf_vectorized_df_ps
tfidf_vectorized_df_ps_h.columns = tfidf_vectorizer_ps.get_feature_names()
tfidf_vectorized_df_ps_h.head()

,,00,000107l,000684l,000710l,001023l,01000sqm,0118651a01,012km,024481i,...,ўвўfrighten,ўвўhave,ўвўin,ўёўў,ўўв,中國理解,在墨尔本的中国婚礼乐队,ﬁnanc,ﬁnancial,ﬁngertip
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
tfidf_vectorized_df_wn_h = tfidf_vectorized_df_wn
tfidf_vectorized_df_wn_h.columns = tfidf_vectorizer_wn.get_feature_names()
tfidf_vectorized_df_wn_h.head()

In [10]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support as score
import xgboost as xgb
import time

In [65]:
print(dir(RandomForestClassifier))
print(RandomForestClassifier())

['__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_cache', '_abc_negative_cache', '_abc_negative_cache_version', '_abc_registry', '_estimator_type', '_get_param_names', '_make_estimator', '_set_oob_score', '_validate_X_predict', '_validate_estimator', '_validate_y_class_weight', 'apply', 'decision_path', 'feature_importances_', 'fit', 'get_params', 'predict', 'predict_log_proba', 'predict_proba', 'score', 'set_params']
RandomForestClassifier(bootstrap=True, class_weight=None, criterion='gini',
            max_depth=None, max_features='auto', max_leaf_nodes=None,
            min_impurity_decrease=0.0,

In [66]:
print(dir(GradientBoostingClassifier))
print(GradientBoostingClassifier())

['_SUPPORTED_LOSS', '__abstractmethods__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_abc_cache', '_abc_negative_cache', '_abc_negative_cache_version', '_abc_registry', '_check_initialized', '_check_params', '_clear_state', '_decision_function', '_estimator_type', '_fit_stage', '_fit_stages', '_get_param_names', '_init_decision_function', '_init_state', '_is_initialized', '_make_estimator', '_resize_state', '_staged_decision_function', '_validate_estimator', '_validate_y', 'apply', 'decision_function', 'feature_importances_', 'fit', 'get_params', 'n_features', 'predict', 'predict_log_proba', 'predict_proba', 'score', 's

## Start to evaluate models

In [11]:
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import train_test_split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(tfidf_vectorized_matrix_ps, data['label'], test_size=0.2)

In [28]:
print(X_train)
print(y_train)

  (0, 12554)	0.897560979187
  (0, 43683)	0.418577057734
  (0, 38357)	0.138482978662
  (1, 546)	0.153514794957
  (1, 25221)	0.140580537403
  (1, 27985)	0.614059179828
  (1, 30343)	0.129450761432
  (1, 31123)	0.116775518268
  (1, 10302)	0.0797108439834
  (1, 7831)	0.176736539174
  (1, 43756)	0.0797108439834
  (1, 1885)	0.0838096269878
  (1, 5411)	0.0679026764305
  (1, 31732)	0.089084966057
  (1, 38273)	0.102735537169
  (1, 44046)	0.0800556182295
  (1, 45950)	0.0960768681014
  (1, 17635)	0.240989165277
  (1, 47144)	0.0874223538907
  (1, 14034)	0.0742547705572
  (1, 41046)	0.106732115752
  (1, 32781)	0.138885720703
  (1, 39241)	0.0787243392195
  (1, 24799)	0.064152586744
  (1, 31790)	0.0704076165584
  :	:
  (31109, 4269)	0.0377460871041
  (31109, 35272)	0.0577218965356
  (31109, 24686)	0.0417189988265
  (31109, 21468)	0.0281246360286
  (31109, 3707)	0.100935388312
  (31109, 14571)	0.0423268753826
  (31109, 46080)	0.0482403344871
  (31109, 11701)	0.0361762110722
  (31109, 2750)	0.0289037009

In [31]:
print(X_test)

  (0, 10272)	0.248899690119
  (0, 44883)	0.248899690119
  (0, 3670)	0.2658434338
  (0, 23632)	0.5316868676
  (0, 23979)	0.210811382518
  (0, 32119)	0.192613791953
  (0, 11749)	0.22008787539
  (0, 25430)	0.222044491763
  (0, 19132)	0.130824386178
  (0, 26435)	0.119020203111
  (0, 14463)	0.18765297304
  (0, 29282)	0.11753117282
  (0, 35354)	0.163801977225
  (0, 31704)	0.106775275246
  (0, 38178)	0.142796360154
  (0, 39919)	0.138101859336
  (0, 5235)	0.220350590333
  (0, 12337)	0.0936864965212
  (0, 15249)	0.12099438661
  (0, 45669)	0.102670656056
  (0, 25754)	0.0851553043744
  (0, 14683)	0.0885781407425
  (0, 11701)	0.0911815794986
  (0, 44489)	0.112900334261
  (0, 21584)	0.073057114076
  :	:
  (7776, 14571)	0.0995837462847
  (7776, 17076)	0.0685402501629
  (7776, 37387)	0.105277003827
  (7776, 30447)	0.0601834483687
  (7776, 40154)	0.0636014563273
  (7776, 26337)	0.284373096009
  (7776, 44750)	0.0794525675833
  (7776, 33974)	0.0831898056697
  (7776, 41667)	0.0833808194948
  (7776, 38357

In [13]:
def train_RF(n_est, depth):
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, n_jobs=-1, verbose=10)
    rf_model = rf.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    precision, recall, fscore, support = score(y_test, y_pred)
    print('Est: {} / Depth: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        n_est, depth, precision, recall,
        round((y_pred==y_test).sum() / len(y_pred), 3)))

In [14]:
from ipywidgets import FloatProgress
from IPython.display import display

In [20]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [10, 50, 100]:
    for depth in [10, 20, 30, None]:
        print("\n\nTraining combination: n_est:{n_est} - depth:{max_depth}".format(n_est=str(n_est),depth=str(depth)))
        progressBar.value+=1
        count += 1
        train_RF(n_est, depth)

A Jupyter Widget



Training combination: n_est:10 - max_depth:10


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 10 / Depth: 10 ---- Precision: [ 0.48833034  0.          0.625       0.          0.          0.          0.64
  0.          0.          0.          0.          0.13306908  0.
  0.73333333  0.          0.          0.          0.          0.
  0.61538462  0.38461538  0.          0.76271186  0.          0.14427481
  0.59447005  0.7         0.          0.          0.          0.625       0.
  0.          0.          1.          0.33333333  0.38271605  0.
  0.68627451  0.          0.          0.          0.53218884  0.
  0.74285714  0.          0.          0.          0.          0.          0.
  0.32142857  0.95454545  0.5         0.          0.          0.
  0.26470588  0.5         0.61111111  0.          0.          0.
  0.28508772  0.          0.          0.          0.          0.
  0.66486486  0.73387097  0.          0.77272727  0.          0.41734417
  0.          0.          0.59183673  0.          0.          0.          0.
  0.          0.          0.55844156  0.          0. 

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


...

Est: 100 / Depth: 20 ---- Precision: [ 0.74100719  0.9375      0.69230769  1.          0.75        0.          0.
  0.          0.          0.          0.73809524  0.22864483  0.          1.
  0.          0.          0.83333333  0.          0.          0.65
  0.91935484  0.          0.8372093   0.          0.23338185  0.79411765
  0.75838926  0.8         0.          0.          0.8375      0.          0.
  0.80952381  0.6         1.          0.32773109  0.          0.77027027
  0.          0.          0.          0.49343832  0.          0.83333333
  0.63157895  0.          0.          0.          0.          0.
  0.54545455  0.          0.52173913  0.          0.5         0.          0.5034965
  0.86666667  0.63432836  0.          0.75        0.          0.32291667
  0.          0.          0.          0.          0.          0.88125
  0.92134831  0.          0.79746835  1.          0.58461538  0.          0.
  0.69026549  0.          0.          0.          0.          0.          0.


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 100 / Depth: 30 ---- Precision: [ 0.81315789  0.86666667  0.58333333  0.75        0.60526316  0.71428571
  0.          0.          0.          0.          0.7         0.27734375
  0.          0.91666667  0.          0.          0.94444444  0.          0.
  0.76470588  0.81578947  0.          0.81203008  1.          0.3423161
  0.78285714  0.66483516  0.875       1.          0.          0.85526316
  0.          0.          0.73333333  0.42857143  0.33333333  0.37223975
  0.          0.78205128  0.5         0.          0.          0.51378446
  0.          0.83333333  0.50666667  0.          0.          0.          0.
  0.          0.5         0.91666667  0.39344262  0.          0.57142857
  0.          0.47945205  0.8         0.64748201  0.          1.          0.
  0.32996633  1.          0.          0.          0.          0.
  0.81818182  0.87735849  0.          0.76190476  0.53333333  0.61621622
  0.          0.          0.74231678  0.          0.          0.          0.4
  0.  

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 100 / Depth: None ---- Precision: [ 0.90144928  0.84210526  0.53191489  0.7037037   0.55555556  0.57692308
  0.72727273  0.83333333  0.          0.          0.64383562  0.4         0.
  0.60869565  0.          0.          0.81967213  1.          0.06666667
  0.68539326  0.82142857  0.          0.77142857  0.875       0.60213618
  0.78494624  0.63768116  0.79545455  1.          0.          0.82954545
  0.18571429  0.4         0.74193548  0.5         0.51724138  0.37412587
  1.          0.625       0.44444444  0.5         0.          0.54061625
  0.          0.77083333  0.54081633  0.4         0.10416667  0.7         0.
  0.          0.37179487  0.95454545  0.37815126  0.          0.5         1.
  0.50980392  0.84615385  0.68055556  1.          0.92857143  0.          0.33
  0.9047619   0.14285714  0.          0.5         0.          0.87058824
  0.88695652  1.          0.78571429  0.37931034  0.69470405  1.
  0.36111111  0.78030303  0.          0.33333333  0.          0.45833333
  

In [31]:
train_RF(900, None) # 0.613 accuracy if i use wordnet lemamtizing

building tree 1 of 900
building tree 4 of 900
building tree 3 of 900
building tree 2 of 900
building tree 5 of 900
building tree 6 of 900
building tree 7 of 900
building tree 8 of 900


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   11.2s


building tree 9 of 900
building tree 10 of 900
building tree 11 of 900
building tree 12 of 900
building tree 13 of 900
building tree 14 of 900
building tree 15 of 900
building tree 16 of 900

[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   17.6s



building tree 17 of 900
building tree 18 of 900
building tree 19 of 900
building tree 20 of 900
building tree 21 of 900


...

building tree 837 of 900
building tree 838 of 900
building tree 839 of 900
building tree 840 of 900
building tree 841 of 900
building tree 842 of 900
building tree 843 of 900
building tree 844 of 900
building tree 845 of 900
building tree 846 of 900
building tree 847 of 900
building tree 848 of 900
building tree 849 of 900
building tree 850 of 900
building tree 851 of 900
building tree 852 of 900
building tree 853 of 900
building tree 854 of 900
building tree 855 of 900
building tree 856 of 900
building tree 857 of 900
building tree 858 of 900
building tree 859 of 900
building tree 860 of 900
building tree 861 of 900
building tree 862 of 900
building tree 863 of 900
building tree 864 of 900
building tree 865 of 900
building tree 866 of 900
building tree 867 of 900
building tree 868 of 900
building tree 869 of 900
building tree 870 of 900
building tree 871 of 900
building tree 872 of 900
building tree 873 of 900
building tree 874 of 900
building tree 875 of 900
building tree 876 of 900


[Parallel(n_jobs=-1)]: Done 874 tasks      | elapsed: 28.2min


building tree 878 of 900
building tree 879 of 900
building tree 880 of 900
building tree 881 of 900
building tree 882 of 900
building tree 883 of 900
building tree 884 of 900
building tree 885 of 900
building tree 886 of 900
building tree 887 of 900
building tree 888 of 900
building tree 889 of 900
building tree 890 of 900
building tree 891 of 900
building tree 892 of 900
building tree 893 of 900
building tree 894 of 900
building tree 895 of 900
building tree 896 of 900
building tree 897 of 900
building tree 898 of 900
building tree 899 of 900
building tree 900 of 900


[Parallel(n_jobs=-1)]: Done 900 out of 900 | elapsed: 28.9min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    1.3s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.1s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    6.5s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    6.9s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    7.2s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    7.4s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:    7.8s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   11.0s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   11.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   12.3s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   12.8s
[Parallel(n_jo

Est: 900 / Depth: None ---- Precision: [ 0.8617284   0.77777778  0.73214286  0.6         0.47826087  0.53731343
  0.6875      1.          0.          0.          0.75862069  0.45292621
  0.          0.73333333  0.          0.          0.89830508  0.375       0.15
  0.61333333  0.84313725  0.          0.86861314  0.875       0.5744089
  0.81656805  0.62264151  0.85106383  0.85714286  0.          0.79487179
  0.21052632  0.44444444  0.75862069  0.39285714  0.63636364  0.35534591
  1.          0.7826087   0.66666667  0.          0.          0.53250774
  0.          0.83333333  0.62765957  0.57142857  0.28        0.6
  0.22222222  0.          0.47142857  0.6875      0.38848921  0.
  0.57894737  0.66666667  0.51369863  0.8         0.62328767  1.          1.
  0.          0.37647059  0.8         0.12        0.          0.63636364
  0.          0.90116279  0.92682927  1.          0.91025641  0.52173913
  0.65373134  0.          0.21568627  0.77925532  0.          0.36363636
  0.          0.64

In [28]:
train_RF(400, None) # 0.618 accuracy if i use wordnet lemamtizing

building tree 1 of 400
building tree 4 of 400building tree 2 of 400building tree 3 of 400


building tree 5 of 400
building tree 6 of 400
building tree 7 of 400
building tree 8 of 400


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   18.8s


building tree 9 of 400
building tree 10 of 400
building tree 11 of 400
building tree 12 of 400
building tree 13 of 400
building tree 14 of 400


[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   27.9s


building tree 15 of 400
building tree 16 of 400
building tree 17 of 400
building tree 18 of 400
building tree 19 of 400
building tree 20 of 400
building tree 21 of 400


...

building tree 362 of 400
building tree 363 of 400
building tree 364 of 400
building tree 365 of 400
building tree 366 of 400
building tree 367 of 400
building tree 368 of 400
building tree 369 of 400
building tree 370 of 400
building tree 371 of 400
building tree 372 of 400
building tree 373 of 400
building tree 374 of 400
building tree 375 of 400
building tree 376 of 400
building tree 377 of 400
building tree 378 of 400
building tree 379 of 400
building tree 380 of 400
building tree 381 of 400
building tree 382 of 400
building tree 383 of 400
building tree 384 of 400
building tree 385 of 400
building tree 386 of 400
building tree 387 of 400
building tree 388 of 400


[Parallel(n_jobs=-1)]: Done 384 tasks      | elapsed: 13.5min


building tree 389 of 400
building tree 390 of 400
building tree 391 of 400
building tree 392 of 400
building tree 393 of 400
building tree 394 of 400
building tree 395 of 400
building tree 396 of 400
building tree 397 of 400
building tree 398 of 400
building tree 399 of 400
building tree 400 of 400


[Parallel(n_jobs=-1)]: Done 400 out of 400 | elapsed: 14.0min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    0.4s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.3s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    1.6s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.3s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    5.4s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    7.4s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    7.8s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    8.2s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:   10.4s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   10.9s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   13.5s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   14.1s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   17.3s
[Parallel(n_jo

Est: 400 / Depth: None ---- Precision: [ 0.89488636  0.875       0.60902256  0.76923077  0.51111111  0.61403509
  0.84210526  0.875       0.          0.33333333  0.72222222  0.45644172
  0.          0.63043478  0.          0.          0.91935484  0.66666667
  0.          0.71578947  0.81818182  0.          0.85034014  0.77777778
  0.57074722  0.79012346  0.65714286  0.9         0.85714286  0.33333333
  0.70786517  0.30081301  0.5         0.88571429  0.48648649  0.48648649
  0.36787565  1.          0.66371681  0.64285714  0.          0.5         0.5
  0.          0.73076923  0.58333333  0.66666667  0.37777778  0.6
  0.22222222  0.          0.3974359   0.94117647  0.41322314  0.
  0.35294118  1.          0.48214286  0.83116883  0.69117647  1.
  0.86363636  0.          0.3768997   0.85        0.24444444  1.
  0.77777778  0.          0.88757396  0.73333333  1.          0.85869565
  0.57142857  0.69069069  1.          0.36111111  0.80869565  0.          0.875
  0.          0.76190476  0.666

In [14]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier()
param = {'n_estimators': [10, 150, 300],
        'max_depth': [30, 90, None],
         'verbose': [10]
        }


In [ ]:
gs = GridSearchCV(rf, param, cv=5, n_jobs=-1, verbose=10)
gs_fit = gs.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(gs_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

Fitting 5 folds for each of 9 candidates, totalling 45 fits


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\model_selection\_split.py:597: Warning: The least populated class in y has only 3 members, which is too few. The minimum number of members in any class cannot be less than n_splits=5.
  % (min_groups, self.n_splits)), Warning)
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   17.2s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  4.2min
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:  8.6min
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed: 21.6min


In [15]:
def train_GB(est, max_depth, lr):
    gb = GradientBoostingClassifier(n_estimators=est, max_depth=max_depth, learning_rate=lr , verbose=10)
    gb_model = gb.fit(X_train, y_train)
    y_pred = gb_model.predict(X_test)
    precision, recall, fscore, train_support = score(y_test, y_pred)
    print('Est: {} / Depth: {} / LR: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        est, max_depth, lr, precision,recall, 
        round((y_pred==y_test).sum()/len(y_pred), 3)))

In [16]:
from ipywidgets import FloatProgress
from IPython.display import display




In [ ]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [50, 100, 150]:
    for max_depth in [3, 7, 11, 15]:
        print("\n\nTraining combination: n_est:{n_est} - max_depth:{max_depth}".format(n_est=str(n_est),max_depth=str(max_depth)))
        progressBar.value+=1
        count += 1
        train_GB(n_est, max_depth, 0.01)

A Jupyter Widget



Training combination: 50 - 3


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)


Est: 50 / Depth: 3 / LR: 0.01 ---- Precision: [ 0.93452381  0.96        0.73451327  0.75        0.28        0.48        0.71875
  0.45        0.          0.41666667  0.81538462  0.34558824  0.
  0.72972973  0.          0.16666667  0.74545455  0.83333333  0.07692308
  0.61458333  0.83333333  0.          0.90833333  0.75        0.65492958
  0.81761006  0.7         0.69565217  0.76470588  0.5         0.81428571
  0.30508475  0.57142857  0.90243902  0.53125     0.53846154  0.28851541
  0.66666667  0.65137615  0.38461538  0.83333333  1.          0.49832776
  0.          0.86956522  0.43434343  0.66666667  0.19354839  0.66666667
  0.08333333  0.55555556  0.40243902  1.          0.4159292   0.          0.53125
  1.          0.55555556  0.81578947  0.8         0.53846154  0.76470588
  0.          0.36394558  0.84615385  0.11363636  0.75        0.59259259
  0.          0.91156463  0.8877551   0.          0.89041096  0.41176471
  0.68543046  0.625       0.18681319  0.77777778  0.          0.6666

Est: 50 / Depth: 15 / LR: 0.01 ---- Precision: [ 0.94864048  0.95238095  0.62601626  0.59375     0.34285714  0.49019608
  0.64        0.64285714  0.          0.          0.77777778  0.27158099
  0.          0.64516129  0.          0.2         0.84444444  0.77777778
  0.03846154  0.52991453  0.79710145  0.          0.90677966  0.8
  0.66793893  0.82550336  0.69871795  0.70212766  0.92857143  0.
  0.81355932  0.17682927  0.5         0.92105263  0.48275862  0.46666667
  0.31481481  0.42857143  0.6         0.30434783  0.83333333  1.          0.4924812
  0.          0.82978723  0.44615385  0.52941176  0.09174312  0.42307692
  0.25        0.14814815  0.32467532  1.          0.37162162  0.
  0.51612903  1.          0.56521739  0.8358209   0.77777778  0.4
  0.90909091  0.          0.37089202  0.8         0.08823529  0.5
  0.45714286  0.07692308  0.93377483  0.91304348  0.25        0.92647059
  0.44444444  0.67054264  0.71428571  0.15789474  0.78456592  0.          0.6
  0.          0.37209302 

In [ ]:
gb = GradientBoostingClassifier()
param = {
    'n_estimators': [50, 100, 150], 
    'max_depth': [7, 11, 15],
    'learning_rate': [0.01]
}

clf = GridSearchCV(gb, param, cv=5, n_jobs=-1)
cv_fit = clf.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

In [ ]:
rf = RandomForestClassifier(n_estimators=150, max_depth=None, n_jobs=-1)
rf_model = rf.fit(tfidf_vectorized_matrix_ps, data['label'])

In [ ]:
QUERY = """ 
SELECT
bm.id
,trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
FROM oneflare_reports.business_master bm
WHERE bm.total_quotes_made < 3
"""

In [ ]:
data_to_predict=pd.read_sql(QUERY,redshift)

In [ ]:
from sklearn.externals import joblib
joblib.dump(rf_model, 'filename.pkl') 
#clf = joblib.load('filename.pk1')

In [15]:
def train_XGB(est, max_depth, lr):
    gb = xgb.XGBClassifier(n_estimators=est, max_depth=max_depth, learning_rate=lr, silent=0, verbose=10)
    gb_model = gb.fit(X_train, y_train)
    y_pred = gb_model.predict(X_test)
    precision, recall, fscore, train_support = score(y_test, y_pred)
    print('Est: {} / Depth: {} / LR: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        est, max_depth, lr, precision,recall, 
        round((y_pred==y_test).sum()/len(y_pred), 3)))

In [16]:
for n_est in [50, 100, 150]:
    for max_depth in [3, 7, 15]:
        print("\n\nTraining combination: n_est:{n_est} - max_depth:{max_depth}".format(n_est=str(n_est),max_depth=str(max_depth)))
        train_XGB(n_est, max_depth, 0.01)



Training combination: n_est:50 - max_depth:3


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 50 / Depth: 3 / LR: 0.01 ---- Precision: [ 0.96048632  0.76923077  0.62962963  0.61538462  0.43478261  0.63291139
  0.64        0.73684211  0.          0.6         0.68656716  0.40526976
  0.          0.56756757  0.          0.          0.81538462  0.8         0.
  0.7032967   0.84810127  0.5         0.91666667  0.72222222  0.65681445
  0.79754601  0.60352423  0.828125    0.66666667  0.4         0.83098592
  0.3         0.44444444  0.8         0.60526316  0.53333333  0.22222222
  0.375       0.71910112  0.40909091  1.          0.          0.53220339
  0.          0.75        0.57647059  0.46153846  0.22972973  0.54545455
  0.          0.33333333  0.54545455  0.76923077  0.49206349  0.          0.35
  0.75        0.47712418  0.79746835  0.75833333  0.64705882  0.71428571
  0.          0.39308176  0.88888889  0.2826087   0.11111111  0.57142857
  0.          0.86746988  0.8989899   0.33333333  0.890625    0.33333333
  0.70212766  0.5         0.2         0.7679558   0.          0.5   

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 50 / Depth: 7 / LR: 0.01 ---- Precision: [ 0.95252226  0.69230769  0.64220183  0.64        0.45454545  0.6375
  0.63636364  0.77777778  0.          0.6         0.62162162  0.45041322
  0.          0.53658537  0.          0.          0.79104478  0.8
  0.33333333  0.67326733  0.83544304  0.33333333  0.93076923  0.72222222
  0.67109635  0.8136646   0.61434978  0.8125      0.66666667  0.4
  0.80821918  0.29927007  0.30769231  0.83333333  0.64864865  0.5106383
  0.28125     0.4         0.69892473  0.52173913  0.6         1.
  0.53559322  0.          0.7826087   0.62352941  0.5         0.31521739
  0.46666667  0.11111111  0.5         0.48235294  0.86956522  0.48837209
  0.          0.29411765  0.75        0.45714286  0.81578947  0.74603175
  0.61111111  0.65217391  0.          0.40229885  0.9         0.248
  0.22222222  0.53846154  0.          0.86549708  0.8989899   0.6         0.875
  0.30769231  0.70716511  0.5         0.25423729  0.78591549  0.          0.5
  0.          0.5        

...

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 150 / Depth: 3 / LR: 0.01 ---- Precision: [ 0.95481928  0.76923077  0.65686275  0.64        0.41071429  0.67948718
  0.625       0.7         0.          0.33333333  0.703125    0.42051282
  0.          0.58974359  0.          0.          0.82539683  0.81818182
  0.          0.7173913   0.85365854  0.25        0.9379845   0.72222222
  0.6650165   0.80368098  0.60619469  0.81538462  0.68        0.28571429
  0.84507042  0.36440678  0.4         0.8         0.66666667  0.58536585
  0.25        0.5         0.75294118  0.56        0.75        0.5
  0.53559322  0.          0.7826087   0.62962963  0.5         0.30337079
  0.46153846  0.          0.          0.54216867  0.86956522  0.48062016
  0.          0.45833333  0.75        0.48366013  0.84        0.76
  0.61111111  0.64        0.          0.39677419  1.          0.28089888
  0.125       0.55555556  0.          0.88271605  0.8989899   0.5
  0.9047619   0.30769231  0.70336391  0.8         0.26315789  0.775       0.
  0.4         0.    

C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)


Est: 150 / Depth: 7 / LR: 0.01 ---- Precision: [ 0.95535714  0.71428571  0.65765766  0.66666667  0.49019608  0.71232877
  0.58333333  0.73684211  0.          0.375       0.63888889  0.46448864
  0.          0.55263158  0.          0.          0.8030303   0.81818182
  0.2         0.68686869  0.85185185  0.25        0.95275591  0.72222222
  0.6677686   0.82208589  0.61607143  0.83870968  0.73913043  0.5
  0.84507042  0.31034483  0.33333333  0.83333333  0.64705882  0.47826087
  0.27472527  0.42857143  0.74157303  0.6         0.66666667  1.
  0.53583618  0.          0.8         0.625       0.53846154  0.28282828
  0.46153846  0.16666667  0.28571429  0.43877551  0.86956522  0.47101449
  0.          0.22222222  0.75        0.48484848  0.83783784  0.76229508
  0.55555556  0.66666667  0.          0.38521401  0.8125      0.24285714
  0.27272727  0.5         0.          0.90853659  0.90721649  0.375
  0.87692308  0.2         0.69592476  0.8         0.29090909  0.78711485
  0.          0.4       

KeyboardInterrupt: 